In [ ]:
import sys
!{sys.executable} -m pip install catboost

!pip list

In [4]:
#basic imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
#modelling
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression   
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings


In [5]:
df=pd.read_csv('StudentsPerformance.csv')

In [6]:
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [7]:
x=df.drop('math score',axis=1)
y=df['math score']

In [8]:
x.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,74
1,female,group C,some college,standard,completed,90,88
2,female,group B,master's degree,standard,none,95,93
3,male,group A,associate's degree,free/reduced,none,57,44
4,male,group C,some college,standard,none,78,75


In [9]:
#create column transformer with 3 types of transformers
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_features=x.select_dtypes(exclude="object").columns
cat_features=x.select_dtypes(include="object").columns

num_transformer=StandardScaler()
cat_transformer=OneHotEncoder(handle_unknown='ignore')

preprocessor=ColumnTransformer(
    [
        ("OneHotEncoder", cat_transformer, cat_features),
        ("StandardScaler", num_transformer, num_features),
    ]
)

In [10]:
x=preprocessor.fit_transform(x) 

In [11]:
x.shape

(1000, 19)

In [12]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)   
x_train.shape,x_test.shape

((800, 19), (200, 19))

Create an Evaluate Function to give all metrics after model_training

In [13]:
def evaluate_model(true, predicted):
    mse=mean_squared_error(true, predicted)
    r2=r2_score(true, predicted)
    mae=mean_absolute_error(true, predicted)
    rmse=np.sqrt(mse)
    return mae,rmse,r2
    

In [20]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

models = {
    "LinearRegression": LinearRegression(),
    "DecisionTreeRegressor": DecisionTreeRegressor(),
    "RandomForestRegressor": RandomForestRegressor(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "KNeighborsRegressor": KNeighborsRegressor(),
    "XGBRegressor": XGBRegressor(),
    "CatBoostRegressor": CatBoostRegressor(verbose=0),
    "AdaBoostRegressor": AdaBoostRegressor()
}
model_list=[]
r2_list=[]
for i in range(len(list(models.keys()))):
    model=list(models.values())[i]
    model.fit(x_train,y_train)#training the model
    
    #make predictions
    y_train_pred=model.predict(x_train)
    y_pred=model.predict(x_test)
    
    #evaluate train and test dataset
    train_mae,train_rmse,train_r2 = evaluate_model(y_train,y_train_pred)
    mae,rmse,r2=evaluate_model(y_test,y_pred)
    
    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])
    
    print("model performance for training set:")
    print(f"Model: {list(models.keys())[i]} \n Train_MAE: {train_mae} \n Train_RMSE: {train_rmse} \n Train_R2 Score: {train_r2}")
    print("----------------------------")
    print("model performance for test set:")
    print(f"Model: {list(models.keys())[i]} \n MAE: {mae} \n RMSE: {rmse} \n R2 Score: {r2}")
    r2_list.append(r2)
    print("-"*50)
    print("\n\n")

LinearRegression
model performance for training set:
Model: LinearRegression 
 Train_MAE: 4.266711846071957 
 Train_RMSE: 5.323050852720514 
 Train_R2 Score: 0.8743172040139593
----------------------------
model performance for test set:
Model: LinearRegression 
 MAE: 4.21476314247485 
 RMSE: 5.393993869732843 
 R2 Score: 0.8804332983749565
--------------------------------------------------



DecisionTreeRegressor
model performance for training set:
Model: DecisionTreeRegressor 
 Train_MAE: 0.01875 
 Train_RMSE: 0.2795084971874737 
 Train_R2 Score: 0.9996534669718089
----------------------------
model performance for test set:
Model: DecisionTreeRegressor 
 MAE: 6.16 
 RMSE: 7.7479029420869745 
 R2 Score: 0.7533065064946594
--------------------------------------------------



RandomForestRegressor
model performance for training set:
Model: RandomForestRegressor 
 Train_MAE: 1.8366760416666668 
 Train_RMSE: 2.3090653933661405 
 Train_R2 Score: 0.9763502220099307
----------------------

Results

In [22]:
result=pd.DataFrame(list(zip(model_list,r2_list)),columns=["Model","R2 Score"]).sort_values(by="R2 Score",ascending=False)
print(result)


                   Model  R2 Score
4                  Ridge  0.880593
0       LinearRegression  0.880433
8      AdaBoostRegressor  0.854212
2  RandomForestRegressor  0.852135
7      CatBoostRegressor  0.851632
6           XGBRegressor  0.827797
3                  Lasso  0.825320
5    KNeighborsRegressor  0.783813
1  DecisionTreeRegressor  0.753307


Linear  Regression

In [ ]:
lin_model=LinearRegress